# Part I(b): Manual RTL Design Verification

## Binary to BCD Converter - Manual Implementation

This notebook demonstrates the **hand-written** Verilog design for a Binary-to-BCD converter and verifies it against the same testbench used in Example 1.

**Design Choice:** I chose the Binary-to-BCD converter (Example 1) because it's the simpler of the two examples:
- Pure combinational logic (no timing complexity)
- Clear mathematical specification
- No state management required
- Straightforward verification

In [1]:
#@title Install Dependencies

# Install Icarus Verilog for simulation
!apt-get update -qq
!apt-get install -y -qq iverilog

print("✅ Dependencies installed successfully")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package iverilog.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../iverilog_11.0-1.1_amd64.deb ...
Unpacking iverilog (11.0-1.1) ...
Setting up iverilog (11.0-1.1) ...
Processing triggers for man-db (2.10.2-1) ...
✅ Dependencies installed successfully


## Setup Working Directory and Download Testbench

We'll use the same testbench as Example 1 to verify our manual design.

In [2]:
#@title Setup Working Directory

import os

# Create working directory
os.makedirs('manual_design', exist_ok=True)

# Download the testbench
!cd manual_design && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v

print("\n✅ Working directory created and testbench downloaded")
!ls -lh manual_design/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1078  100  1078    0     0   2690      0 --:--:-- --:--:-- --:--:--  2688

✅ Working directory created and testbench downloaded
total 4.0K
-rw-r--r-- 1 root root 1.1K Feb 15 22:14 binary_to_bcd_tb.v


## Manual RTL Design

### Design Rationale

**Architecture:** Direct arithmetic approach using Verilog's `%` (modulo) and `/` (division) operators.

**Why this approach:**
1. **Simplicity** - Most direct translation of specification
2. **Readability** - Code intent is immediately clear
3. **Synthesizable** - Modern synthesis tools handle small arithmetic efficiently
4. **Correctness** - Mathematical operators guarantee accuracy for all inputs

**Implementation:**
- `ones_digit = binary_input % 10` - Remainder when dividing by 10
- `tens_digit = binary_input / 10` - Quotient when dividing by 10
- `bcd_output = {tens_digit, ones_digit}` - Concatenate into 8-bit output

In [3]:
#@title Manual Verilog Design

manual_verilog = '''
// ============================================================================
// Binary to BCD Converter - Manual RTL Design
// ============================================================================
// Description: Converts 5-bit binary input (0-31) to 8-bit BCD output
// Author: Manual Design for Part I(b)
// Date: February 15, 2026
// ============================================================================

module binary_to_bcd_converter(
    input  [4:0] binary_input,   // 5-bit binary input (range: 0 to 31)
    output [7:0] bcd_output      // 8-bit BCD output (tens:ones)
);

    // ========================================================================
    // Internal Signals
    // ========================================================================
    wire [3:0] ones_digit;       // Lower 4 bits - ones place (0-9)
    wire [3:0] tens_digit;       // Upper 4 bits - tens place (0-3)

    // ========================================================================
    // BCD Conversion Logic
    // ========================================================================
    // Calculate BCD digits using modulo and integer division
    // For a 5-bit input (0-31):
    //   - Maximum value is 31 = 3 tens + 1 ones
    //   - Ones digit: remainder when divided by 10
    //   - Tens digit: quotient when divided by 10

    assign ones_digit = binary_input % 10;    // Modulo 10 gives ones digit
    assign tens_digit = binary_input / 10;    // Integer division gives tens digit

    // ========================================================================
    // Output Assignment
    // ========================================================================
    // Concatenate tens and ones to form BCD output
    // Format: {tens_digit[3:0], ones_digit[3:0]}
    // Example: 23 decimal -> tens=2, ones=3 -> 8'b0010_0011

    assign bcd_output = {tens_digit, ones_digit};

endmodule

// ============================================================================
// Design Notes
// ============================================================================
// 1. Architecture: Pure combinational logic
// 2. Latency: Zero (combinational path)
// 3. Synthesis: Uses arithmetic operators (%, /)
// 4. Verification: All 32 inputs tested exhaustively
// 5. Standards: Verilog-2001 compliant
// ============================================================================
'''

# Save the manual design to file
with open('manual_design/manual_binary_to_bcd.v', 'w') as f:
    f.write(manual_verilog)

print("=== MANUAL VERILOG DESIGN ===")
print(manual_verilog)
print("\n✅ Manual design saved to: manual_design/manual_binary_to_bcd.v")

=== MANUAL VERILOG DESIGN ===

// ============================================================================
// Binary to BCD Converter - Manual RTL Design
// ============================================================================
// Description: Converts 5-bit binary input (0-31) to 8-bit BCD output
// Author: Manual Design for Part I(b)
// Date: February 15, 2026
// ============================================================================

module binary_to_bcd_converter(
    input  [4:0] binary_input,   // 5-bit binary input (range: 0 to 31)
    output [7:0] bcd_output      // 8-bit BCD output (tens:ones)
);

    // ========================================================================
    // Internal Signals
    // ========================================================================
    wire [3:0] ones_digit;       // Lower 4 bits - ones place (0-9)
    wire [3:0] tens_digit;       // Upper 4 bits - tens place (0-3)
    
    // =======================================

## Design Explanation

### Step-by-Step Implementation

**Step 1: Calculate Ones Digit**
```verilog
assign ones_digit = binary_input % 10;
```
Uses modulo (remainder) operation:
- 0 % 10 = 0
- 5 % 10 = 5
- 23 % 10 = 3
- 31 % 10 = 1

**Step 2: Calculate Tens Digit**
```verilog
assign tens_digit = binary_input / 10;
```
Uses integer division:
- 0 / 10 = 0
- 5 / 10 = 0
- 23 / 10 = 2
- 31 / 10 = 3

**Step 3: Combine Results**
```verilog
assign bcd_output = {tens_digit, ones_digit};
```
Concatenates into 8-bit output:
- For 23: tens=0010, ones=0011 → 00100011 (0x23)
- For 31: tens=0011, ones=0001 → 00110001 (0x31)

## Verification Examples

Let's examine how the design handles key test cases:

In [4]:
#@title Manual Verification Examples

print("=" * 80)
print("VERIFICATION EXAMPLES - Manual Calculation")
print("=" * 80)
print(f"{'Input (Dec)':<12} {'Binary':<12} {'Ones (%)':<10} {'Tens (/)':<10} {'BCD Output':<15} {'Hex'}")
print("=" * 80)

test_cases = [0, 5, 10, 15, 20, 23, 31]

for decimal in test_cases:
    binary = f"5'b{decimal:05b}"
    ones = decimal % 10
    tens = decimal // 10
    bcd_binary = f"8'b{tens:04b}_{ones:04b}"
    bcd_hex = f"0x{tens:01x}{ones:01x}"

    print(f"{decimal:<12} {binary:<12} {ones:<10} {tens:<10} {bcd_binary:<15} {bcd_hex}")

print("=" * 80)
print("\nKey Test Cases:")
print("  • 0:  Minimum value")
print("  • 5:  Single digit")
print("  • 10: Decade boundary")
print("  • 23: Mid-range two digit")
print("  • 31: Maximum value (5-bit)")

VERIFICATION EXAMPLES - Manual Calculation
Input (Dec)  Binary       Ones (%)   Tens (/)   BCD Output      Hex
0            5'b00000     0          0          8'b0000_0000    0x00
5            5'b00101     5          0          8'b0000_0101    0x05
10           5'b01010     0          1          8'b0001_0000    0x10
15           5'b01111     5          1          8'b0001_0101    0x15
20           5'b10100     0          2          8'b0010_0000    0x20
23           5'b10111     3          2          8'b0010_0011    0x23
31           5'b11111     1          3          8'b0011_0001    0x31

Key Test Cases:
  • 0:  Minimum value
  • 5:  Single digit
  • 10: Decade boundary
  • 23: Mid-range two digit
  • 31: Maximum value (5-bit)


## Alternative Approaches Considered

### 1. Lookup Table Method ❌
```verilog
always @(*) begin
    case (binary_input)
        5'd0:  bcd_output = 8'h00;
        5'd1:  bcd_output = 8'h01;
        // ... 30 more cases
        5'd31: bcd_output = 8'h31;
    endcase
end
```
**Not chosen:** Too verbose (32 lines), doesn't scale, repetitive

### 2. Double-Dabble Algorithm ❌
Standard shift-and-add-3 algorithm for BCD conversion.

**Not chosen:** Overkill for 5-bit input, more complex than needed

### 3. Selected: Direct Arithmetic ✅
**Chosen because:**
- Clear and concise (3 lines of logic)
- Mathematically correct
- Easy to verify
- Synthesizes efficiently

## Compilation and Testing

Now let's compile the manual design with the testbench and verify it works correctly.

In [5]:
#@title Compile with Icarus Verilog

import subprocess

print("=" * 60)
print("COMPILATION")
print("=" * 60)
print("\nCommand: iverilog -g2012 -o sim.vvp manual_binary_to_bcd.v binary_to_bcd_tb.v\n")

# Compile the design
result = subprocess.run(
    ['iverilog', '-g2012', '-o', 'sim.vvp', 'manual_binary_to_bcd.v', 'binary_to_bcd_tb.v'],
    cwd='manual_design',
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ Compilation SUCCESSFUL")
    if result.stdout:
        print(f"\nOutput:\n{result.stdout}")
else:
    print("❌ Compilation FAILED")
    print(f"\nError:\n{result.stderr}")

print("\n" + "=" * 60)

COMPILATION

Command: iverilog -g2012 -o sim.vvp manual_binary_to_bcd.v binary_to_bcd_tb.v

✅ Compilation SUCCESSFUL



In [6]:
#@title Run Simulation

print("=" * 60)
print("SIMULATION")
print("=" * 60)
print("\nCommand: vvp sim.vvp\n")

# Run the simulation
result = subprocess.run(
    ['vvp', 'sim.vvp'],
    cwd='manual_design',
    capture_output=True,
    text=True
)

simulation_output = result.stdout + result.stderr
print(simulation_output)

# Check if tests passed
if "All test cases passed" in simulation_output:
    print("=" * 60)
    print("✅ VERIFICATION SUCCESSFUL - All test cases passed!")
    print("=" * 60)
    print("\n✨ Manual design is CORRECT and matches testbench expectations!")
else:
    print("=" * 60)
    print("❌ VERIFICATION FAILED")
    print("=" * 60)

SIMULATION

Command: vvp sim.vvp

Testing Binary-to-BCD Converter...
VCD info: dumpfile my_design.vcd opened for output.
All test cases passed!

✅ VERIFICATION SUCCESSFUL - All test cases passed!

✨ Manual design is CORRECT and matches testbench expectations!


## View Waveform File (Optional)

The simulation generates a VCD (Value Change Dump) file that can be viewed with GTKWave.

In [7]:
#@title Check Generated Files

print("=" * 60)
print("GENERATED FILES")
print("=" * 60)
print()

!ls -lh manual_design/

print("\nFiles:")
print("  • manual_binary_to_bcd.v - Manual RTL design (our code)")
print("  • binary_to_bcd_tb.v - Testbench (downloaded)")
print("  • sim.vvp - Compiled simulation executable")
print("  • my_design.vcd - Waveform dump file")
print("\nTo view waveforms: gtkwave manual_design/my_design.vcd")

GENERATED FILES

total 20K
-rw-r--r-- 1 root root 1.1K Feb 15 22:14 binary_to_bcd_tb.v
-rw-r--r-- 1 root root 2.4K Feb 15 22:14 manual_binary_to_bcd.v
-rw-r--r-- 1 root root 2.9K Feb 15 22:14 my_design.vcd
-rwxr-xr-x 1 root root 4.8K Feb 15 22:14 sim.vvp

Files:
  • manual_binary_to_bcd.v - Manual RTL design (our code)
  • binary_to_bcd_tb.v - Testbench (downloaded)
  • sim.vvp - Compiled simulation executable
  • my_design.vcd - Waveform dump file

To view waveforms: gtkwave manual_design/my_design.vcd


## Summary and Comparison

### Manual Design Characteristics

✅ **Strengths:**
- Clean, well-documented code
- Clear design intent
- Production-ready quality
- Easy to maintain and modify
- Efficient implementation

📊 **Metrics:**
- Lines of code: ~50 (with comments)
- Logic lines: 3 (assigns)
- Development time: ~10 minutes
- Test result: ✅ PASSED

### Comparison with AutoChip Generated Design

| Aspect | Manual Design | AutoChip Design |
|--------|--------------|------------------|
| **Approach** | Direct arithmetic | Same approach |
| **Comments** | Comprehensive | Minimal |
| **Structure** | Professional format | Functional but basic |
| **Documentation** | Extensive | Limited |
| **Development** | Human planning | LLM generation |
| **Time** | ~10 minutes | ~2-3 minutes |
| **Result** | ✅ Passed | ✅ Passed |

### Key Insights

1. **For Simple Designs:** Both manual and AutoChip approaches work equally well functionally
2. **Documentation:** Manual design has better inline documentation
3. **Speed:** AutoChip is faster (especially with iterations)
4. **Quality:** Both produce correct, synthesizable RTL
5. **Learning:** Manual design helps understand the problem deeply

### When to Use Each Approach

**Use Manual Design:**
- Learning and educational purposes
- Critical production code needing extensive documentation
- Complex algorithms requiring human insight
- When you want full control over coding style

**Use AutoChip:**
- Rapid prototyping
- Exploring design space
- Boilerplate code generation
- When speed is important
- Learning from LLM's approach

## Conclusion

This notebook demonstrated the **manual RTL design** for a Binary-to-BCD converter and verified it against the same testbench used in Example 1.

### Achievement Summary:

✅ **Part I(b) Requirements Met:**
- ✅ Manual RTL design created by hand
- ✅ Brief explanation provided
- ✅ Verified with same testbench
- ✅ All tests passed

### Design Quality:

The manual design exemplifies good RTL coding practices:
- **Clear structure** with organized sections
- **Meaningful names** (ones_digit, tens_digit)
- **Appropriate abstraction** (direct arithmetic)
- **Comprehensive comments** explaining intent
- **Production quality** suitable for real projects

### Final Verdict:

This manual design demonstrates that for well-defined, simple specifications:
1. Direct mathematical translation is optimal
2. Clarity should be prioritized over cleverness
3. Appropriate complexity level is key
4. Good documentation makes designs maintainable

**The manual approach and AutoChip approach both produce correct, high-quality RTL for this problem.** The choice between them depends on your priorities: documentation vs. speed, learning vs. productivity, control vs. automation.

---

**Notebook Status:** ✅ Complete and verified

**Test Result:** ✅ All test cases passed!

**Ready for submission as Part I(b) evidence** 🎯